In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, RFE

from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, roc_auc_score, precision_recall_curve, auc)


In [ ]:
df = pd.read_csv(r'../merge/raw/csv_full.csv', sep=';')

null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

pd.DataFrame({
    'null_count': null_counts,
    'null_%': null_pct
}).sort_values('null_%', ascending=False)

In [ ]:
df_model = df.drop(columns=['nom_commune', 'dep_nom', 'reg_nom'])

df_model.head()

In [ ]:
df_model.columns.to_list()

In [ ]:
features = ['annee', 'code_commune', 'dep_code', 'reg_code', 'code_postal', 'population', 'densite', 'superficie_km2',
 'zone_emploi', 'taux_global_tfb', 'taux_global_tfnb', 'taux_plein_teom', 'taux_global_th']

X = df_model[features]
y = df_model['y']

train = df_model[df_model['annee'] < 2024]
test  = df_model[df_model['annee'] == 2024]

X_train = train[features]
y_train = train['y']
X_test  = test[features]
y_test  = test['y']

print(f"Train : {len(X_train)} lignes")
print(f"Test  : {len(X_test)} lignes")
print(f"\nDistribution train :\n{y_train.value_counts()}")
print(f"\nDistribution test :\n{y_test.value_counts()}")

In [ ]:
pipeline_simple = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=5)),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

pipeline_simple.fit(X_train, y_train)

# Prédictions
y_pred = pipeline_simple.predict(X_test)

# Évaluation
accuracy = accuracy_score(y_test, y_pred)
print(f"✓ Pipeline entraîné !")
print(f"Accuracy : {accuracy:.2%}")

In [ ]:
print(classification_report(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred, labels=['hausse', 'baisse', 'stable'])
